# Linear Probe — concept detection on residual stream

Fit a linear probe on any HF model's residual stream to detect a user-specified concept from `(text, label)` pairs. This notebook produces the **indispensable baseline** against which every SAE feature-pack AUROC claim must be measured.

**Why this matters.** A dictionary-learning method (SAE, crosscoder, MLP-probing, etc.) only earns its keep if a *feature-pack* (sparse subset of its features) beats what a plain logistic regression already extracts from the raw residual stream. Without the probe baseline, every "we found a feature for X" claim is unfalsifiable.

**Citations**
- Alain & Bengio 2016, *Understanding intermediate layers using linear classifier probes*, arXiv:1610.01644 — founding paper.
- Belrose et al. 2023, *Eliciting Latent Predictions from Transformers with the Tuned Lens*, arXiv:2303.08112 — modern per-layer sweep.
- Apollo Research 2025, `probity` — reference library pattern (github.com/ApolloResearch/probity).
- Farquhar et al. 2023, *Challenges with unsupervised LLM knowledge discovery*, arXiv:2312.10029 — why **difference-of-means** must be reported alongside logistic probes.
- Huben et al. 2024 (SAE feature evaluation) and Karvonen et al. 2025 (SAEBench) — probe-AUROC-as-baseline is the convention SAE work is held to.

**What we report per layer**: logistic regression (plain), logistic regression (standardised features), and difference-of-means. The winning layer + probe vector is dumped to `probe_results.json` for downstream use.

In [ ]:
!pip install -q transformers==4.57.1 accelerate safetensors scikit-learn==1.6.1 pandas matplotlib tqdm

## Config + data

Point `CSV_PATH` at a CSV with two columns: `text` (string) and `label` (binary 0/1 or multi-class int/string). If the file is missing, we fall back to a 400-sample synthetic positive/negative review corpus so the notebook is runnable end-to-end out of the box.

In [ ]:
import os, json, random, gc, pathlib
import numpy as np
import pandas as pd
import torch

MODEL_ID = 'google/gemma-2-2b'
LAYERS_TO_SWEEP = [4, 8, 12, 16, 20]   # 5 evenly spaced layers for Gemma-2-2b (26 layers total)
CSV_PATH = 'train.csv'                 # columns: 'text', 'label'
TEST_SPLIT = 0.2
RANDOM_SEED = 42
SEQ_LEN = 256
POOL_LAST_K = 8                        # mean-pool over last K non-padding tokens
BATCH_SIZE = 8
CACHE_DIR = pathlib.Path('./probe_cache')
CACHE_DIR.mkdir(exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def make_synthetic(n_per_class=200):
    pos_templates = [
        'I absolutely loved this {noun}, it was {adj_pos}.',
        'Best {noun} I have used in years, truly {adj_pos}.',
        'Five stars: {adj_pos} {noun}, highly recommended.',
        'The {noun} exceeded my expectations, genuinely {adj_pos}.',
        'What a {adj_pos} {noun} — could not be happier.',
    ]
    neg_templates = [
        'I regret buying this {noun}, it was {adj_neg}.',
        'Worst {noun} ever, completely {adj_neg}.',
        'Do not buy this {noun}: {adj_neg} and broken.',
        'The {noun} disappointed me, frankly {adj_neg}.',
        'Awful {noun} — {adj_neg} and a waste of money.',
    ]
    nouns = ['gadget', 'book', 'film', 'laptop', 'coffee maker', 'restaurant', 'hotel', 'game']
    adj_pos = ['fantastic', 'delightful', 'superb', 'brilliant', 'outstanding']
    adj_neg = ['terrible', 'dreadful', 'shoddy', 'dismal', 'broken']
    rows = []
    for _ in range(n_per_class):
        t = random.choice(pos_templates).format(noun=random.choice(nouns), adj_pos=random.choice(adj_pos))
        rows.append({'text': t, 'label': 1})
    for _ in range(n_per_class):
        t = random.choice(neg_templates).format(noun=random.choice(nouns), adj_neg=random.choice(adj_neg))
        rows.append({'text': t, 'label': 0})
    random.shuffle(rows)
    return pd.DataFrame(rows)

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded {len(df)} rows from {CSV_PATH}')
else:
    df = make_synthetic(200)
    print(f'{CSV_PATH} not found — using synthetic demo corpus ({len(df)} rows)')

assert 'text' in df.columns and 'label' in df.columns, 'CSV must have columns: text, label'

# Encode labels as contiguous ints
label_values = sorted(df['label'].unique().tolist())
label_to_idx = {v: i for i, v in enumerate(label_values)}
df['y'] = df['label'].map(label_to_idx).astype(int)
N_CLASSES = len(label_values)
print(f'Classes ({N_CLASSES}): {label_values}')
print(df.head())

## Load model + tokenize

Load in `bfloat16` with SDPA attention (no flash-attn). We tokenize with left-padding disabled and truncate to `SEQ_LEN=256` — probe work is typically insensitive to long context, and this keeps the run ≤ 10 min on a T4.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map=device,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# Infer depth + width
N_LAYERS = getattr(model.config, 'num_hidden_layers', None) or len(model.model.layers)
D_MODEL = getattr(model.config, 'hidden_size', None) or model.model.layers[0].self_attn.o_proj.out_features
print(f'{MODEL_ID}: num_hidden_layers={N_LAYERS}, hidden_size={D_MODEL}')

# Guard against bad layer indices
LAYERS_TO_SWEEP = [L for L in LAYERS_TO_SWEEP if 0 <= L <= N_LAYERS]
print(f'Sweeping layers: {LAYERS_TO_SWEEP}')

texts = df['text'].astype(str).tolist()
labels = df['y'].to_numpy()

enc = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=SEQ_LEN,
    return_tensors='pt',
)
print('input_ids shape:', tuple(enc['input_ids'].shape))

## Extract residual-stream activations per layer

For each requested layer, grab `hidden_states[layer]` and mean-pool over the last `POOL_LAST_K` non-padding tokens per sequence. Result per layer: `(N, D_MODEL)` float32 numpy array. Cached to `./probe_cache/` so repeated probe fits don't re-run the model.

In [ ]:
from tqdm.auto import tqdm

def last_k_mean(hidden, attn_mask, k):
    """hidden: (B, T, D); attn_mask: (B, T). Mean over last k non-pad tokens per row."""
    B, T, D = hidden.shape
    lengths = attn_mask.sum(dim=1)  # (B,)
    out = torch.zeros(B, D, dtype=hidden.dtype, device=hidden.device)
    for i in range(B):
        L = int(lengths[i].item())
        if L == 0:
            continue
        start = max(0, L - k)
        out[i] = hidden[i, start:L].mean(dim=0)
    return out

def extract_layer_activations(layers, batch_size=BATCH_SIZE):
    acts = {L: [] for L in layers}
    input_ids = enc['input_ids']
    attn_mask = enc['attention_mask']
    n = input_ids.shape[0]
    max_layer = max(layers)
    for s in tqdm(range(0, n, batch_size), desc='forward'):
        e = min(n, s + batch_size)
        ids = input_ids[s:e].to(device)
        am = attn_mask[s:e].to(device)
        with torch.no_grad():
            out = model(
                input_ids=ids,
                attention_mask=am,
                output_hidden_states=True,
                use_cache=False,
            )
        hs = out.hidden_states  # tuple of (B, T, D), length N_LAYERS+1 (incl. embeddings)
        for L in layers:
            pooled = last_k_mean(hs[L], am, POOL_LAST_K)
            acts[L].append(pooled.float().cpu().numpy())
        del out, hs
    for L in layers:
        acts[L] = np.concatenate(acts[L], axis=0)
    return acts

cache_file = CACHE_DIR / f'acts_{MODEL_ID.replace("/", "_")}_{"-".join(map(str, LAYERS_TO_SWEEP))}_k{POOL_LAST_K}_s{SEQ_LEN}_n{len(texts)}.npz'
if cache_file.exists():
    print(f'Loading cached activations: {cache_file}')
    z = np.load(cache_file)
    activations = {int(k.split('_')[1]): z[k] for k in z.files}
else:
    activations = extract_layer_activations(LAYERS_TO_SWEEP)
    np.savez_compressed(cache_file, **{f'layer_{L}': A for L, A in activations.items()})
    print(f'Cached to {cache_file}')

for L, A in activations.items():
    print(f'layer {L:2d}: activations shape {A.shape}')

# Free VRAM before probe fitting
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Fit probes + sweep layers

For every layer we report three probes side-by-side:

1. **Logistic Regression (L2, `C=1e-3`)** — the textbook linear probe.
2. **Logistic Regression with feature-scaling** (`StandardScaler` inside the CV pipeline) — guards against magnitude-dominant dims.
3. **Difference-of-means** — Farquhar 2023 insists this is reported alongside any logistic probe because it is hyperparameter-free and frequently matches or beats trained probes; failure to report it has tripped up multiple unsupervised-probing papers.

Metric: 5-fold stratified CV **AUROC** (binary; macro one-vs-rest for multi-class) plus accuracy and macro-F1. Winning `(layer, method)` is picked by CV AUROC.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

def auroc_multi(y_true, scores):
    """Binary AUROC from (N,) scores; multi-class macro OvR AUROC from (N, C) probs."""
    if scores.ndim == 1:
        return roc_auc_score(y_true, scores)
    return roc_auc_score(y_true, scores, multi_class='ovr', average='macro')

def fit_logreg(X, y, scale=False):
    base = LogisticRegression(
        C=1e-3, max_iter=1000, class_weight='balanced', solver='lbfgs',
        multi_class='auto',
    )
    clf = Pipeline([('scaler', StandardScaler()), ('lr', base)]) if scale else base
    # Per-fold AUROC
    aurocs = []
    for tr, te in skf.split(X, y):
        clf.fit(X[tr], y[tr])
        if N_CLASSES == 2:
            s = clf.predict_proba(X[te])[:, 1]
        else:
            s = clf.predict_proba(X[te])
        aurocs.append(auroc_multi(y[te], s))
    # Cross-validated predictions for acc/F1
    y_pred = cross_val_predict(clf, X, y, cv=skf, method='predict')
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average='macro')
    # Final fit on all data → export probe weights
    clf.fit(X, y)
    return {
        'auroc_mean': float(np.mean(aurocs)),
        'auroc_std': float(np.std(aurocs)),
        'acc': float(acc),
        'f1': float(f1),
        'model': clf,
    }

def fit_diffmeans(X, y):
    """Binary: project onto (mu_pos - mu_neg). Multi-class: class-mean vs rest mean, macro OvR."""
    aurocs_per_fold = []
    for tr, te in skf.split(X, y):
        Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]
        if N_CLASSES == 2:
            mu1 = Xtr[ytr == 1].mean(axis=0)
            mu0 = Xtr[ytr == 0].mean(axis=0)
            w = mu1 - mu0
            scores = Xte @ w
            aurocs_per_fold.append(roc_auc_score(yte, scores))
        else:
            per_class_auc = []
            for c in range(N_CLASSES):
                mu_c = Xtr[ytr == c].mean(axis=0)
                mu_rest = Xtr[ytr != c].mean(axis=0)
                w = mu_c - mu_rest
                scores = Xte @ w
                y_bin = (yte == c).astype(int)
                if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
                    per_class_auc.append(roc_auc_score(y_bin, scores))
            aurocs_per_fold.append(float(np.mean(per_class_auc)) if per_class_auc else float('nan'))
    # Final fit on all data for export
    if N_CLASSES == 2:
        W = (X[y == 1].mean(axis=0) - X[y == 0].mean(axis=0))[None, :]  # (1, D)
    else:
        W = np.stack([X[y == c].mean(axis=0) - X[y != c].mean(axis=0) for c in range(N_CLASSES)], axis=0)
    # acc/F1 via simple cross-val predictions
    y_pred_all = np.zeros_like(y)
    for tr, te in skf.split(X, y):
        if N_CLASSES == 2:
            w = X[tr][y[tr] == 1].mean(0) - X[tr][y[tr] == 0].mean(0)
            y_pred_all[te] = (X[te] @ w > 0).astype(int)
        else:
            Wf = np.stack([X[tr][y[tr] == c].mean(0) - X[tr][y[tr] != c].mean(0) for c in range(N_CLASSES)], axis=0)
            y_pred_all[te] = (X[te] @ Wf.T).argmax(axis=1)
    return {
        'auroc_mean': float(np.nanmean(aurocs_per_fold)),
        'auroc_std': float(np.nanstd(aurocs_per_fold)),
        'acc': float(accuracy_score(y, y_pred_all)),
        'f1': float(f1_score(y, y_pred_all, average='macro')),
        'W': W,
    }

results = {}
for L in LAYERS_TO_SWEEP:
    X = activations[L].astype(np.float32)
    y = labels.astype(int)
    r_lr = fit_logreg(X, y, scale=False)
    r_lr_scaled = fit_logreg(X, y, scale=True)
    r_dm = fit_diffmeans(X, y)
    results[L] = {'logreg': r_lr, 'logreg_scaled': r_lr_scaled, 'diffmeans': r_dm}
    print(
        f'layer {L:2d} | '
        f"logreg AUROC={r_lr['auroc_mean']:.3f}±{r_lr['auroc_std']:.3f} "
        f"acc={r_lr['acc']:.3f} f1={r_lr['f1']:.3f} | "
        f"scaled AUROC={r_lr_scaled['auroc_mean']:.3f} | "
        f"diff-means AUROC={r_dm['auroc_mean']:.3f}"
    )

# Pick winner across (layer, method)
best = None
for L, r in results.items():
    for method, rr in r.items():
        score = rr['auroc_mean']
        if best is None or score > best['auroc']:
            best = {'layer': L, 'method': method, 'auroc': score}
print(f"\nBEST: layer {best['layer']} / {best['method']} → AUROC {best['auroc']:.4f}")

## Compare probe modes

Side-by-side table of all three methods across all swept layers. If difference-of-means matches or beats plain logreg, that is the finding to report — per Farquhar et al. 2023, an SAE/probe method that only beats logreg but not diff-means is not demonstrating real capability.

In [ ]:
rows = []
for L in LAYERS_TO_SWEEP:
    r = results[L]
    rows.append({
        'layer': L,
        'logreg_auroc': r['logreg']['auroc_mean'],
        'logreg_scaled_auroc': r['logreg_scaled']['auroc_mean'],
        'diffmeans_auroc': r['diffmeans']['auroc_mean'],
        'logreg_acc': r['logreg']['acc'],
        'logreg_f1': r['logreg']['f1'],
    })
comp = pd.DataFrame(rows).set_index('layer')
print(comp.round(4).to_string())

# Crude "who-wins-per-layer"
winners = comp[['logreg_auroc', 'logreg_scaled_auroc', 'diffmeans_auroc']].idxmax(axis=1)
print('\nWinner per layer:')
print(winners.to_string())

## Visualise the per-layer sweep

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
xs = LAYERS_TO_SWEEP
for method, label, marker in [
    ('logreg', 'LogReg (L2, C=1e-3)', 'o'),
    ('logreg_scaled', 'LogReg + StandardScaler', 's'),
    ('diffmeans', 'Difference-of-means', '^'),
]:
    ys = [results[L][method]['auroc_mean'] for L in xs]
    errs = [results[L][method]['auroc_std'] for L in xs]
    ax.errorbar(xs, ys, yerr=errs, label=label, marker=marker, capsize=3, linewidth=1.5)

ax.axvline(best['layer'], color='red', linestyle='--', alpha=0.5, label=f"best: L{best['layer']}/{best['method']}")
ax.axhline(0.5, color='grey', linestyle=':', alpha=0.5, label='chance (binary)')
ax.set_xlabel('Layer index (residual stream)')
ax.set_ylabel('5-fold CV AUROC')
ax.set_title(f'Linear probe sweep on {MODEL_ID}\nN={len(texts)}, classes={N_CLASSES}, seq_len={SEQ_LEN}')
ax.set_ylim(0.4, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('probe_sweep.png', dpi=150)
plt.show()
print('Saved probe_sweep.png')

In [ ]:
# Dump per-layer metrics + best probe weights
out = {
    'model_id': MODEL_ID,
    'n_samples': int(len(texts)),
    'n_classes': int(N_CLASSES),
    'label_values': [str(v) for v in label_values],
    'layers_swept': LAYERS_TO_SWEEP,
    'pool_last_k': POOL_LAST_K,
    'seq_len': SEQ_LEN,
    'seed': RANDOM_SEED,
    'best': best,
    'per_layer': {
        str(L): {
            m: {k: v for k, v in r.items() if k not in ('model', 'W')}
            for m, r in results[L].items()
        }
        for L in LAYERS_TO_SWEEP
    },
}
with open('probe_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print('Wrote probe_results.json')

# Extract + save best probe weights as a plain numpy array
bL, bM = best['layer'], best['method']
if bM == 'diffmeans':
    W = results[bL]['diffmeans']['W']  # (C_or_1, D)
    b = np.zeros(W.shape[0], dtype=np.float32)
else:
    clf = results[bL][bM]['model']
    if isinstance(clf, Pipeline):
        lr = clf.named_steps['lr']
        scaler = clf.named_steps['scaler']
        # Fold scaler into the linear weights so inference is a pure (x @ W.T + b)
        W_raw = lr.coef_                                      # (C_or_1, D)
        b_raw = lr.intercept_                                 # (C_or_1,)
        mu, sigma = scaler.mean_, scaler.scale_
        W = W_raw / sigma                                     # absorb scale
        b = (b_raw - (W_raw * (mu / sigma)).sum(axis=1)).astype(np.float32)
    else:
        W = clf.coef_
        b = clf.intercept_
W = np.asarray(W, dtype=np.float32)
b = np.asarray(b, dtype=np.float32)
np.savez('probe_weights.npz', W=W, b=b, layer=bL, method=bM, label_values=np.array(label_values, dtype=object))
print(f'Wrote probe_weights.npz  (W shape={W.shape}, b shape={b.shape}, layer={bL}, method={bM})')

print('\n--- usage snippet ---')
print(f"""
# At inference time, given residual-stream activation `h` of shape (D,) from layer {bL}:
#   h = hidden_states[{bL}][last_k_mean]  # same pooling as training
# Binary:
#   logit = float(h @ W[0] + b[0]); prob = 1/(1+np.exp(-logit))
# Multi-class:
#   logits = h @ W.T + b; probs = softmax(logits)
#   pred_label = label_values[int(np.argmax(logits))]
# SAE baseline rule: any SAE feature-pack claiming to detect this concept
# must beat AUROC = {best['auroc']:.4f} on a held-out split of the same data.
""".strip())